In [1]:
import numpy as np
import lal
import taichi as ti

ti.init(
    arch=ti.cpu,
    default_fp=ti.f64,
    cpu_max_num_threads=1,
    offline_cache=False,
    debug=True,
)

import jax
jax.config.update("jax_enable_x64", True)

from matplotlib import pyplot as plt

%matplotlib inline

from pespace.detector.orbit import (
    KaplerianHeliocentric,
    KeplerianGeocentric,
    ConstellationVectorStruct,
)

/tmp/ipykernel_113233/1761044830.py:2: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
[I 07/06/26 17:17:20.891 113233] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


[Taichi] version 1.7.4, llvm 15.0.4, commit b4b956fd, linux, python 3.12.13
[Taichi] Starting on arch=x64


In [2]:
tsamples_np = np.arange(0, lal.YRJUL_SI, 3600)
tsamples = ti.field(ti.f64, shape=tsamples_np.shape)
tsamples.from_numpy(tsamples_np)

orb_helio_ti = KaplerianHeliocentric(2.5e9, 0.0, 0.0, backend='taichi')
orb_geo_ti = KeplerianGeocentric(1.0e8, 0.0, 0.0, backend='taichi')

orb_vec_container_helio = ConstellationVectorStruct.field(shape=tsamples_np.shape)
orb_vec_container_geo = ConstellationVectorStruct.field(shape=tsamples_np.shape)

AU_SEC = lal.AU_SI / lal.C_SI

@ti.kernel
def orbit_test():
    for i in tsamples:
        orb_vec_container_helio[i] = orb_helio_ti.get_constellation_vectors(tsamples[i])
        orb_vec_container_geo[i] = orb_geo_ti.get_constellation_vectors(tsamples[i])

orbit_test()

data_helio_ti = orb_vec_container_helio.to_numpy()
data_geo_ti = orb_vec_container_geo.to_numpy()

In [3]:
orb_helio_jax = KaplerianHeliocentric(2.5e9, 0.0, 0.0, backend='jax')
orb_geo_jax = KeplerianGeocentric(1.0e8, 0.0, 0.0, backend='jax')

data_helio_jax = jax.jit(jax.vmap(orb_helio_jax.get_constellation_vectors))(tsamples_np)
data_geo_jax = jax.jit(jax.vmap(orb_geo_jax.get_constellation_vectors))(tsamples_np)


In [4]:
np.testing.assert_allclose(data_helio_jax.n1, data_helio_ti['n1'])
np.testing.assert_allclose(data_helio_jax.n2, data_helio_ti['n2'], rtol=1e-10, atol=1e-10)
np.testing.assert_allclose(data_helio_jax.n3, data_helio_ti['n3'])
np.testing.assert_allclose(data_helio_jax.x1, data_helio_ti['x1'])
np.testing.assert_allclose(data_helio_jax.x2, data_helio_ti['x2'])
np.testing.assert_allclose(data_helio_jax.x3, data_helio_ti['x3'])

In [5]:
np.testing.assert_allclose(data_geo_jax.n1, data_geo_ti['n1'])
np.testing.assert_allclose(data_geo_jax.n2, data_geo_ti['n2'])
np.testing.assert_allclose(data_geo_jax.n3, data_geo_ti['n3'])
np.testing.assert_allclose(data_geo_jax.x1, data_geo_ti['x1'])
np.testing.assert_allclose(data_geo_jax.x2, data_geo_ti['x2'])
np.testing.assert_allclose(data_geo_jax.x3, data_geo_ti['x3'])